In [66]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy import units as u
from astropy import constants as const

df = pd.read_csv("solar_system.csv")
print("BEFORE transpose shape:", df.shape)
df.head()


BEFORE transpose shape: (20, 11)


,Attribute,Mercury,Venus,Earth,Moon,Mars,Jupiter,Saturn,Uranus,Neptune,Pluto
0,Mass (10^24kg),0.330,4.87,5.97,0.073,0.642,1898,568,86.8,102,0.0130
1,Diameter (km),4879,12104,12756,3475,6792,142984,120536,51118,49528,2376
2,Density (kg/m^3),5429,5243,5514,3340,3934,1326,687,1270,1638,1850
3,Gravity (m/s^2),3.7,8.9,9.8,1.6,3.7,23.1,9.0,8.7,11.0,0.7
4,Escape Velocity (km/s),4.3,10.4,11.2,2.4,5.0,59.5,35.5,21.3,23.5,1.3


In [67]:
print("Start shape:", df.shape)

df1 = df.set_index("Attribute")
print("After set_index('Attribute'):", df1.shape)

df2 = df1.T
print("After transpose (.T):", df2.shape)

df2.index.name = "Planet"
print("After df.index.name = 'Planet':", df2.shape)

df3 = df2.reset_index()
print("After reset_index():", df3.shape)

df3.columns.name = None
print("After df.columns.name = None:", df3.shape)

dfT = df3
dfT.head()
print("Before:", df.shape, " -> rows =", df.shape[0], "cols =", df.shape[1])
print("After: ", dfT.shape, " -> rows =", dfT.shape[0], "cols =", dfT.shape[1])
print(dfT.columns.tolist())
import re

def has_unit(colname: str) -> bool:
    return bool(re.search(r"\(.*\)|\[.*\]", str(colname)))

unit_cols = [c for c in dfT.columns if has_unit(c)]
no_unit_cols = [c for c in dfT.columns if not has_unit(c)]

print("Columns WITH units:", len(unit_cols))
print(unit_cols)

print("\nColumns WITHOUT units:", len(no_unit_cols))
print(no_unit_cols)
print(dfT)


Start shape: (20, 11)
After set_index('Attribute'): (20, 10)
After transpose (.T): (10, 20)
After df.index.name = 'Planet': (10, 20)
After reset_index(): (10, 21)
After df.columns.name = None: (10, 21)
Before: (20, 11)  -> rows = 20 cols = 11
After:  (10, 21)  -> rows = 10 cols = 21
['Planet', 'Mass (10^24kg)', 'Diameter (km)', 'Density (kg/m^3)', 'Gravity (m/s^2)', 'Escape Velocity (km/s)', 'Rotation Period (hours)', 'Length of Day (hours)', 'Distance from Sun (10^6 km)', 'Perihelion (10^6 km)', 'Aphelion (10^6 km)', 'Orbital Period (days)', 'Orbital Velocity (km/s)', 'Orbital Inclination (deg)', 'Orbital Eccentricity', 'Obliquity to Orbit (deg)', 'Mean Temperature (C)', 'Surface Pressure (bars)', 'Number of Moons', 'Ring System?', 'Global Magnetic Field?']
Columns WITH units: 16
['Mass (10^24kg)', 'Diameter (km)', 'Density (kg/m^3)', 'Gravity (m/s^2)', 'Escape Velocity (km/s)', 'Rotation Period (hours)', 'Length of Day (hours)', 'Distance from Sun (10^6 km)', 'Perihelion (10^6 km)', 

In [68]:
df.dtypes

Attribute    object
Mercury      object
Venus        object
Earth        object
Moon         object
Mars         object
Jupiter      object
Saturn       object
Uranus       object
Neptune      object
Pluto        object
dtype: object

In [69]:
for col in df.columns:
    print(col, df[col].apply(type).value_counts().head(5))
    print("-----")
# a) most of the values here are strings, and some colmns may contain booleans. 
# b) Because strings don't behave like numbers , we cannot compare or do anything related with plotting. 

Attribute Attribute
<class 'str'>    20
Name: count, dtype: int64
-----
Mercury Mercury
<class 'str'>    20
Name: count, dtype: int64
-----
Venus Venus
<class 'str'>    20
Name: count, dtype: int64
-----
Earth Earth
<class 'str'>    20
Name: count, dtype: int64
-----
Moon Moon
<class 'str'>    20
Name: count, dtype: int64
-----
Mars Mars
<class 'str'>    20
Name: count, dtype: int64
-----
Jupiter Jupiter
<class 'str'>    20
Name: count, dtype: int64
-----
Saturn Saturn
<class 'str'>    20
Name: count, dtype: int64
-----
Uranus Uranus
<class 'str'>    20
Name: count, dtype: int64
-----
Neptune Neptune
<class 'str'>    20
Name: count, dtype: int64
-----
Pluto Pluto
<class 'str'>    20
Name: count, dtype: int64
-----


In [76]:
skip_cols = ["Planet", "Ring System?", "Global Magnetic Field?"]

for col in df.columns:
    if col not in skip_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
df.dtypes
# a) the three different data types we have are float64, boolean, and object.
# b) float64: decimal numeric values, bool: True/False values, object: mixed/non-numeric Python objects, usually strings 
# c) the columns that are still object are Planet, Ring System, and Global Magnetic Field,
# because they contain text and categorical values, not pure numeric data, so they should not be converted to numbers.

Attribute    float64
Mercury      float64
Venus        float64
Earth        float64
Moon         float64
Mars         float64
Jupiter      float64
Saturn       float64
Uranus       float64
Neptune      float64
Pluto        float64
dtype: object

In [84]:
# I want to rebuild the df, because things are a little messy
import numpy as np
import pandas as pd

df = pd.read_csv("solar_system.csv")

df = df.set_index("Attribute").T
df.index.name = "Planet"
df = df.reset_index()
df.columns.name = None
skip_cols = ["Planet", "Ring System?", "Global Magnetic Field?"]
for col in df.columns:
    if col not in skip_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print(df.columns.tolist())


['Planet', 'Mass (10^24kg)', 'Diameter (km)', 'Density (kg/m^3)', 'Gravity (m/s^2)', 'Escape Velocity (km/s)', 'Rotation Period (hours)', 'Length of Day (hours)', 'Distance from Sun (10^6 km)', 'Perihelion (10^6 km)', 'Aphelion (10^6 km)', 'Orbital Period (days)', 'Orbital Velocity (km/s)', 'Orbital Inclination (deg)', 'Orbital Eccentricity', 'Obliquity to Orbit (deg)', 'Mean Temperature (C)', 'Surface Pressure (bars)', 'Number of Moons', 'Ring System?', 'Global Magnetic Field?']


In [87]:
# 2.2
import numpy as np
import pandas as pd
from astropy import units as u
from astropy.units import Quantity

def attach_units(col_name):
    if col_name not in df.columns:
        print("Column not found:", col_name)
        return

    factor = 1.0
    new_name = col_name

    if col_name.startswith("Mass"):
        unit = u.kg
        factor = 1e24 if "10^24" in col_name else 1.0
        new_name = "Mass (kg)"

    elif col_name.startswith("Distance from Sun"):
        unit = u.km
        factor = 1e6 if "10^6" in col_name else 1.0
        new_name = "Distance from Sun (km)"

    elif col_name.startswith("Perihelion"):
        unit = u.km
        factor = 1e6 if "10^6" in col_name else 1.0
        new_name = "Perihelion (km)"

    elif col_name.startswith("Aphelion"):
        unit = u.km
        factor = 1e6 if "10^6" in col_name else 1.0
        new_name = "Aphelion (km)"

    elif col_name == "Diameter (km)":
        unit = u.km
        factor = 1.0
        new_name = "Diameter (km)"

    elif col_name.startswith("Orbital Period"):
        unit = u.day
        factor = 1.0
        new_name = "Orbital Period (days)"

    else:
        print("No unit rule for:", col_name)
        return

    def to_quantity(x):
        if isinstance(x, Quantity):
            return x.to(unit)

        x = pd.to_numeric(x, errors="coerce")
        if pd.isna(x):
            return np.nan * unit
        return (x * factor) * unit

    df[col_name] = df[col_name].apply(to_quantity)

    if new_name != col_name:
        df.rename(columns={col_name: new_name}, inplace=True)

for c in ["Mass (10^24kg)", "Diameter (km)",
          "Distance from Sun (10^6 km)", "Perihelion (10^6 km)", "Aphelion (10^6 km)",
          "Orbital Period (days)"]:
    if c in df.columns:
        attach_units(c)

for c in ["Mass (kg)", "Distance from Sun (km)", "Perihelion (km)", "Aphelion (km)", "Orbital Period (days)"]:
    if c in df.columns:
        attach_units(c)

print(df.columns.tolist())
print("Example type:", type(df["Perihelion (km)"].dropna().iloc[0]) if "Perihelion (km)" in df.columns else "missing")


['Planet', 'Mass (kg)', 'Diameter (km)', 'Density (kg/m^3)', 'Gravity (m/s^2)', 'Escape Velocity (km/s)', 'Rotation Period (hours)', 'Length of Day (hours)', 'Distance from Sun (km)', 'Perihelion (km)', 'Aphelion (km)', 'Orbital Period (days)', 'Orbital Velocity (km/s)', 'Orbital Inclination (deg)', 'Orbital Eccentricity', 'Obliquity to Orbit (deg)', 'Mean Temperature (C)', 'Surface Pressure (bars)', 'Number of Moons', 'Ring System?', 'Global Magnetic Field?']
Example type: <class 'astropy.units.quantity.Quantity'>


In [88]:
# 2.3.1
peri_col = "Perihelion (km)"
aphe_col = "Aphelion (km)"

semi_major = (df[peri_col] + df[aphe_col]) / 2

new_col = "Semi-Major Axis (km)"
if new_col in df.columns:
    df.drop(columns=[new_col], inplace=True)

idx = df.columns.get_loc(aphe_col) + 1
df.insert(idx, new_col, semi_major)

print(df.columns.tolist())
print(df[new_col])
print("Example value type:", type(df[new_col].dropna().iloc[0]))


['Planet', 'Mass (kg)', 'Diameter (km)', 'Density (kg/m^3)', 'Gravity (m/s^2)', 'Escape Velocity (km/s)', 'Rotation Period (hours)', 'Length of Day (hours)', 'Distance from Sun (km)', 'Perihelion (km)', 'Aphelion (km)', 'Semi-Major Axis (km)', 'Orbital Period (days)', 'Orbital Velocity (km/s)', 'Orbital Inclination (deg)', 'Orbital Eccentricity', 'Obliquity to Orbit (deg)', 'Mean Temperature (C)', 'Surface Pressure (bars)', 'Number of Moons', 'Ring System?', 'Global Magnetic Field?']
0      57900000.0 km
1     108200000.0 km
2     149600000.0 km
3        384500.0 km
4     228000000.0 km
5     778500000.0 km
6    1432050000.0 km
7    2867050000.0 km
8    4515000000.0 km
9    5906350000.0 km
Name: Semi-Major Axis (km), dtype: object
Example value type: <class 'astropy.units.quantity.Quantity'>


In [95]:
# 2.3.2
import numpy as np
import pandas as pd
from astropy import units as u
from astropy.units import Quantity

orbital_cols = [c for c in df.columns if "orbital" in str(c).lower() and "period" in str(c).lower()]
print("Orbital-period-like columns I see:", orbital_cols)

days_col = next((c for c in orbital_cols if "day" in str(c).lower()), None)
years_existing_col = next((c for c in orbital_cols if "year" in str(c).lower()), None)

new_col = "Orbital Period (years)"

def to_years_cell(x):
    if isinstance(x, Quantity):
        return x.to(u.yr)
    x = pd.to_numeric(x, errors="coerce")
    if pd.isna(x):
        return np.nan * u.yr
    return (x * u.day).to(u.yr)

if days_col is None:
    if years_existing_col is not None:
        print(f"Looks already in years as column: {years_existing_col}")
        if years_existing_col != new_col:
            df.rename(columns={years_existing_col: new_col}, inplace=True)
    else:
        raise KeyError("I can't find any orbital period column (neither days nor years).")
else:
    years_series = df[days_col].apply(to_years_cell)
    idx = df.columns.get_loc(days_col)

    if new_col in df.columns:
        df.drop(columns=[new_col], inplace=True)

    df.insert(idx, new_col, years_series)
    df.drop(columns=[days_col], inplace=True)

print(df.columns.tolist())
print(df[new_col])

fav = "Earth"
q = df.loc[df["Planet"] == fav, new_col].iloc[0]
val = q.to_value(u.yr) if isinstance(q, Quantity) else float(q)
print(f"{fav} orbital period = {val:.4f} years")


Orbital-period-like columns I see: ['Orbital Period (years)']
Looks already in years as column: Orbital Period (years)
['Planet', 'Mass (kg)', 'Diameter (AU)', 'Density (kg/m^3)', 'Gravity (m/s^2)', 'Escape Velocity (km/s)', 'Rotation Period (hours)', 'Length of Day (hours)', 'Distance from Sun (AU)', 'Perihelion (AU)', 'Aphelion (AU)', 'Semi-Major Axis (AU)', 'Orbital Period (years)', 'Orbital Velocity (km/s)', 'Orbital Inclination (deg)', 'Orbital Eccentricity', 'Obliquity to Orbit (deg)', 'Mean Temperature (C)', 'Surface Pressure (bars)', 'Number of Moons', 'Ring System?', 'Global Magnetic Field?']
0    0.24093086926762491 yr
1     0.6151950718685831 yr
2     0.9998631074606433 yr
3    0.07474332648870637 yr
4     1.8809034907597535 yr
5      11.85763175906913 yr
6     29.423682409308693 yr
7      83.74811772758385 yr
8     163.72347707049965 yr
9     247.93976728268308 yr
Name: Orbital Period (years), dtype: object
Earth orbital period = 0.9999 years


In [94]:
# 2.3.3
from astropy import constants as const
from astropy.units import Quantity

print("AU = the average distance between Earth and the Sun.")

print("1 AU (astropy) =", const.au)

print("1 AU in km =", const.au.to(u.km))

km_cols = [c for c in df.columns if str(c).strip().endswith("(km)")]

for col in km_cols:
    idx = df.columns.get_loc(col)
    new_col = col.replace("(km)", "(AU)")

    def km_to_au(x):
        if isinstance(x, Quantity):
            return x.to(u.au)
        x = pd.to_numeric(x, errors="coerce")
        if pd.isna(x):
            return np.nan * u.au
        return (x * u.km).to(u.au)

    converted = df[col].apply(km_to_au)

    if new_col in df.columns:
        df.drop(columns=[new_col], inplace=True)

    df.insert(idx, new_col, converted)
    df.drop(columns=[col], inplace=True)

print(df.columns.tolist())

fav = "Earth"
row = df.loc[df["Planet"] == fav].iloc[0]
au_cols = [c for c in df.columns if str(c).strip().endswith("(AU)")]

for c in au_cols:
    q = row[c]
    val = q.to_value(u.au) if isinstance(q, Quantity) else float(q)
    print(f"{fav} {c} = {val:.6f} AU")


AU = the average distance between Earth and the Sun.
1 AU (astropy) =   Name   = Astronomical Unit
  Value  = 149597870700.0
  Uncertainty  = 0.0
  Unit  = m
  Reference = IAU 2012 Resolution B2
1 AU in km = 149597870.70000002 km
['Planet', 'Mass (kg)', 'Diameter (AU)', 'Density (kg/m^3)', 'Gravity (m/s^2)', 'Escape Velocity (km/s)', 'Rotation Period (hours)', 'Length of Day (hours)', 'Distance from Sun (AU)', 'Perihelion (AU)', 'Aphelion (AU)', 'Semi-Major Axis (AU)', 'Orbital Period (years)', 'Orbital Velocity (km/s)', 'Orbital Inclination (deg)', 'Orbital Eccentricity', 'Obliquity to Orbit (deg)', 'Mean Temperature (C)', 'Surface Pressure (bars)', 'Number of Moons', 'Ring System?', 'Global Magnetic Field?']
Earth Diameter (AU) = 0.000085 AU
Earth Distance from Sun (AU) = 1.000014 AU
Earth Perihelion (AU) = 0.983303 AU
Earth Aphelion (AU) = 1.016726 AU
Earth Semi-Major Axis (AU) = 1.000014 AU


In [96]:
df.to_csv("units.csv", index=False)